# G1 Academy Bonus - Task 4: robot-state observation (dictionary-based get_* interface)

## Introduction
This task builds the dictionary-based state readers described in `notes.txt` section 2: `get_lowstate`, `get_odommodestate`, `get_battery`, `get_slam_info`, `get_occupancygrid`, `get_rgbd`, and `get_services`. Each one wraps a native subscriber or SDK client and normalizes the result into a plain dictionary, so higher-level tasks never touch raw DDS message layouts directly. This notebook is self-contained: it repeats the `ensure_channel_factory`/`Latest` helpers from Task 2 so it can run on its own.

In [ ]:
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from unitree_sdk2py.core.channel import ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_
import sys
if ".." not in sys.path: sys.path.append("..")
from sdk_wrapper import ensure_channel_factory

ensure_channel_factory(0, "eth0")

class Latest:
    # TODO: cache a newest DDS message and receipt timestamp; add fresh(max_age_s).
    pass

def wait_for_latest(latest, timeout_s=3.0, max_age_s=1.0):
    # TODO: wait briefly for a fresh message, returning None if nothing arrives.
    raise NotImplementedError

# TODO: implement small DataFrame display helpers for None/dict/list results.


## Task 1 - `get_lowstate()`
Joint positions/velocities/torques and IMU fields, read from the cached `rt/lowstate` message.

In [ ]:
lowstate_sub = Latest("rt/lowstate", LowState_)

def get_lowstate(timeout_s=3.0):
    # TODO: return timestamp, q/dq/tau_est arrays, and IMU rpy/gyro/acc as a dict.
    raise NotImplementedError

# TODO: implement a compact display_lowstate(data), then call it.


## Task 2 - `get_odommodestate()`
Field names on `SportModeState_` vary a little across SDK/firmware versions, so read defensively with `getattr` fallbacks instead of assuming one exact attribute name - the same defensive pattern `sdk_wrapper` uses internally.

In [ ]:
from unitree_sdk2py.idl.unitree_go.msg.dds_ import SportModeState_
odom_sub = Latest("rt/odommodestate", SportModeState_)

def _first_attr(obj, names, default=None):
    # TODO: return the first available attribute name for SDK-version compatibility.
    raise NotImplementedError

def get_odommodestate(timeout_s=3.0):
    # TODO: return position, velocity, mode, and gait_type using defensive attribute lookup.
    raise NotImplementedError


## Task 3 - `get_battery()`
Battery data can arrive on a dedicated `BmsState_` topic (name and message module vary by platform), or embedded inside `rt/lowstate`. Try the dedicated topics first, fall back to `rt/lowstate` fields.

In [ ]:
import importlib

def _bms_types():
    # TODO: discover BmsState_ in the Unitree HG/GO IDL modules.
    raise NotImplementedError

BMS_TOPICS = ["rt/lf/bmsstate", "rt/lf/agvbmsstate", "rt/bmsstate", "rt/agvbmsstate"]
# TODO: create subscribers and implement get_battery(), preferring fresh dedicated BMS data.


## Task 4 - `get_slam_info()`
SLAM status arrives as a JSON-in-`String_` payload on `rt/slam_info`, with `rt/slam_key_info` as a fallback.

In [ ]:
from unitree_sdk2py.idl.std_msgs.msg.dds_ import String_
slam_info_sub = Latest("rt/slam_info", String_)
slam_key_sub = Latest("rt/slam_key_info", String_)

def get_slam_info(timeout_s=3.0):
    # TODO: read the primary JSON string, then fall back to slam_key_info.
    raise NotImplementedError


## Task 5 - `get_occupancygrid()` / lidar map plot
The deployed map-like grid available on this robot is `rt/utlidar/map_state` (`HeightMap_`). Treat it as an occupancy-style grid for observation: wait for a fresh message, reshape the flat data into a 2-D array, display the metadata as a DataFrame, and plot the grid with Matplotlib.

In [ ]:
from unitree_sdk2py.idl.unitree_go.msg.dds_ import HeightMap_
occupancygrid_sub = Latest("rt/utlidar/map_state", HeightMap_, queue_len=5)

def get_occupancygrid(timeout_s=3.0):
    # TODO: validate width, height, resolution, and data length before reshaping the map grid.
    raise NotImplementedError


## Task 6 - `get_rgbd()`
RGB-D frames come from `rgbd_server_service` (see `realsense_rgbd_zmq_stream.py`), a ZMQ PUB socket sending `[rgb_jpeg, depth_png, depth_scale]` multipart frames. Connect as SUB, take the newest frame, and decode the pieces lazily (callers decide whether they need the depth channel).

In [ ]:
import os, struct

def get_rgbd(endpoints=None):
    # TODO: subscribe over ZMQ, receive RGB/depth/scale multipart data, and close each socket.
    raise NotImplementedError

# TODO: decode and display the RGB image and optional depth image.


## Task 7 - `get_services()`
`RobotStateClient.ServiceList()` returns every registered service's name/status/protect flag; annotate the known ones with the documented description catalog.

In [ ]:
try:
    from unitree_sdk2py.b2.robot_state.robot_state_client import RobotStateClient
except ImportError:
    from unitree_sdk2py.go2.robot_state.robot_state_client import RobotStateClient

# TODO: initialise RobotStateClient, define SERVICE_CATALOG, and implement get_services().


### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.